# 04 — Predictions & Hypothesis Testing

The core loop:
1. Backtest hypotheses on historical data to establish baselines
2. Generate pre-race predictions for upcoming races
3. Resolve predictions after the race
4. Review the scorecard — what's working, what isn't
5. Adjust and repeat

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt

from src.data import JolpicaClient
from src.predictions import (
    PredictionRegistry,
    Scorecard,
    Backtester,
    RacePredictor,
)

%matplotlib inline

## 1. Load Historical Data

In [ ]:
jolpica = JolpicaClient()

# Load results from 2018-2024
results = jolpica.get_all_results(start_year=2018, end_year=2024)
print(f'Loaded {len(results)} race results across {results["season"].nunique()} seasons')
print(f'Circuits: {results["circuitId"].nunique()}')
print(f'Drivers: {results["driverId"].nunique()}')

## 2. Backtest All Hypotheses

Train on 2018-2022, test on 2023-2024. This tells us which hypotheses
have predictive power before we trust them with real predictions.

In [ ]:
registry = PredictionRegistry()
backtester = Backtester(registry=registry)

backtest_results = backtester.run_all(results, train_end_year=2022)

for name, df in backtest_results.items():
    if not df.empty:
        acc = df['correct'].mean()
        avg_conf = df['confidence'].mean()
        print(f'{name}: {acc:.1%} accuracy ({len(df)} races), avg confidence {avg_conf:.1%}')

In [ ]:
# Detailed pole winner results
pole_results = backtest_results.get('pole_winner', pd.DataFrame())
if not pole_results.empty:
    print('Pole Winner Hypothesis — Race by Race:')
    for _, row in pole_results.iterrows():
        mark = '✓' if row['correct'] else '✗'
        print(f"  {mark} {row['season']} R{row['round']:02d} {row['gp_name']}: {row['details']}")

## 3. Scorecard — How Are We Doing?

In [ ]:
scorecard = Scorecard(registry)

# Summary table
print('=== Hypothesis Accuracy ===')
print(scorecard.summary().to_string())

print('\n=== Calibration ===')
print(scorecard.calibration_table().to_string())

In [ ]:
# Visual scorecard
fig = scorecard.plot_dashboard()
plt.show()

## 4. Predict an Upcoming Race

Example: predict the next race on the calendar.

In [ ]:
# Initialize predictor with full historical data
predictor = RacePredictor(historical_results=results, registry=registry)

# Get qualifying results for the upcoming race
# (Replace with actual qualifying data when available)
quali = jolpica.get_qualifying(2024, 1)  # Example: 2024 Bahrain

# Build pre-race data
pre_race = predictor.build_pre_race_data(
    qualifying_results=quali,
    circuit_id='bahrain',
    race_laps=57,
    weather_forecast={
        'rain_probability': 0.05,
        'track_temp': 32.0,
        'weather_change_risk': 0.05,
    },
)

print('Pre-race data keys:', list(pre_race.keys()))

In [ ]:
# Generate predictions
predictions = predictor.predict_race(
    season=2024,
    round_num=1,
    gp_name='Bahrain',
    pre_race_data=pre_race,
)

# Display predictions
for name, pred in predictions.items():
    conf = pred.get('confidence', 0)
    bar = '█' * int(conf * 10) + '░' * (10 - int(conf * 10))
    print(f'\n{name.upper()}')
    print(f'  Prediction: {pred.get("value", "?")}')
    print(f'  Confidence: {bar} {conf:.0%}')
    print(f'  Reasoning:  {pred.get("reasoning", "")}')

## 5. After the Race — Resolve Predictions

Once the race is complete, feed in the actual outcomes to see how we did.

In [ ]:
# Get actual race results
race_results = jolpica.get_race_results(2024, 1)
pit_stops = jolpica.get_pit_stops(2024, 1)

# Build outcomes
outcomes = predictor.build_post_race_outcomes(
    race_results=race_results,
    pit_stops=pit_stops,
    had_safety_car=True,   # you'd know this from race control data
    had_rain=False,
)

# Resolve
resolution = predictor.resolve_race(2024, 1, outcomes)

print('\n=== Resolution ===')
for name, result in resolution.items():
    mark = '✓' if result['correct'] else '✗'
    print(f'{mark} {name}: correct={result["correct"]}')

In [ ]:
# Updated scorecard
print(predictor.season_report())

## 6. Season-Level Analysis

After multiple races, look at trends — which hypotheses are improving,
which are getting worse, and which need recalibration.

In [ ]:
# Process all 2024 races in sequence
for rnd in range(1, 25):
    try:
        quali = jolpica.get_qualifying(2024, rnd)
        race = jolpica.get_race_results(2024, rnd)
        pits = jolpica.get_pit_stops(2024, rnd)
        
        if quali.empty or race.empty:
            continue
        
        circuit = race['circuitId'].iloc[0]
        gp_name = race['raceName'].iloc[0]
        
        pre_race = predictor.build_pre_race_data(
            qualifying_results=quali,
            circuit_id=circuit,
        )
        
        predictor.predict_race(2024, rnd, gp_name, pre_race)
        
        outcomes = predictor.build_post_race_outcomes(
            race_results=race,
            pit_stops=pits,
        )
        predictor.resolve_race(2024, rnd, outcomes)
        
    except Exception as e:
        print(f'R{rnd}: {e}')

# Final season report
print(predictor.season_report())

In [ ]:
# Final dashboard
fig = scorecard.plot_dashboard()
plt.show()